# Simplified Chinese support and reach on Steam

**DATA2010 Data Science Programming · final project**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theonlypgk/data2010-steam-chinese/blob/main/steam_chinese_analysis.ipynb)

**Main question:** should a Steam game support Simplified Chinese, for which genres, and when?
Reach = number of reviews. This notebook is the whole project, from the two raw input files
to the answers. Run it from top to bottom and it rebuilds every output table.

| Part | What it does | Output tables |
| :--- | :--- | :--- |
| [Setup](#scrollTo=setup) | connect the runtime to the project folder and check the package versions | |
| [Step 1 · Clean](#scrollTo=step1) | load the two snapshots, clean them, filter and encode | `pcd_games_clean` |
| [Step 2 · SQ1 groups](#scrollTo=step2) | group similar games, with vs without Chinese | `pcd_sq1_groups` |
| [Step 3 · SQ2 groups](#scrollTo=step3) | group similar games, added Chinese after launch vs did not | `pcd_sq2_groups` |
| [Save and check](#scrollTo=save) | write the three tables, stop with an error if anything is wrong | |
| [Part B · Analysis](#scrollTo=partb) | figures and insights for EDA, SQ1, SQ2, SQ3 (reads the saved tables only) | `report/figures/` |

Each table is saved whole as `data/processed/csv/pcd_<table>.csv` (to read in code) and
`data/processed/xlsx/pcd_<table>.xlsx` (to open and look at).

### How to run

- **Google Colab:** click *Open in Colab*, then *Runtime → Run all*. Setup clones the
  repository into the runtime and installs the pinned package versions. On a new runtime it
  stops once and asks for *Runtime → Restart session*; after that, *Run all* runs to the end.
  Files written on Colab stay in the runtime and disappear when it is deleted.
- **Own computer or Codespaces:** Python 3.12 or 3.13, `pip install -r requirements.txt`, open
  the notebook from the repository folder and choose *Run All*, or run
  `jupyter nbconvert --to notebook --execute steam_chinese_analysis.ipynb --inplace`.

**Rules for a reproducible run**
- Run the cells in order, from Setup down. To repeat any part, start again from Setup.
- The notebook is the only thing that writes the files in `data/processed/`; they are never
  edited by hand.
- Every run gives the same output tables: the inputs are checked against their SHA-256
  fingerprints and the package versions are pinned. The CSV files are byte-identical on every
  machine; the xlsx files hold identical content (only their zip compression depends on the
  machine's zlib library).

**How to read it.** The markdown at the top of each step explains the strategy and why each
choice was made. Each fixed value is set in the code cell that first uses it, with the reason
in a comment. The first line of each code cell sums up what the cell does. Problems found in
the real data are marked `DATA ISSUE`. The **Check** notes after a result give the numbers a
correct run shows; the checks in *Save and check* stop the notebook if any of them differs.

## Setup · Connect the runtime to the project

Run these two cells first, on any machine.

- **Project folder.** The notebook looks for the repository folder from where it is opened.
  On Colab, which starts in an empty `/content` folder, it clones the repository there first.
  It then moves into the repository folder, so every path below means the same thing on every
  machine.
- **Package versions.** The installed versions are compared with `requirements.txt`. The same
  code can give different results with a different pandas or numpy version, so a mismatch
  stops the notebook; on Colab the pinned versions are installed instead. `jupyter` and
  `streamlit` are not checked: they open the notebook and the app, the analysis does not use
  them.

In [43]:
# Find the project folder (clone it first on Colab) and check the package versions.
import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/theonlypgk/data2010-steam-chinese"
IN_COLAB = "google.colab" in sys.modules           # True when the notebook runs on Google Colab
NOT_NEEDED_TO_RUN = {"jupyter", "streamlit"}        # open the notebook / the app; not imported here
TESTED_PYTHON = {(3, 12), (3, 13)}                  # versions on which the pins were run end to end

if sys.version_info[:2] not in TESTED_PYTHON:
    raise RuntimeError(f"Python {sys.version.split()[0]}: use Python 3.12 or 3.13 (requirements.txt pins).")


def find_root():
    """First folder, from here upwards, that holds requirements.txt and this notebook."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "requirements.txt").exists() and (folder / "steam_chinese_analysis.ipynb").exists():
            return folder
    return None


ROOT = find_root()
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/data2010-steam-chinese")
    if not ROOT.exists():
        # GIT_LFS_SKIP_SMUDGE=1: leave the two large input files out of the clone;
        # Step 1.1 fetches them from the release, which has no Git LFS download quota.
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True,
                       env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
if ROOT is None:
    raise FileNotFoundError(f"Open the notebook inside the repository folder (git clone {REPO_URL}).")
os.chdir(ROOT)                                      # every relative path now starts here


def pinned_versions() -> dict:
    """Package -> exact version from requirements.txt, for the packages this notebook imports."""
    pins = {}
    for line in (ROOT / "requirements.txt").read_text().splitlines():
        line = line.split("#")[0].strip()          # drop the comments
        if "==" in line:
            name, version = (part.strip() for part in line.split("=="))
            if name not in NOT_NEEDED_TO_RUN:
                pins[name] = version
    return pins


def installed_version(name: str):
    """Installed version of a package, or None if it is not installed."""
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


PINS = pinned_versions()
wrong = {name: pin for name, pin in PINS.items() if installed_version(name) != pin}
if wrong and IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[f"{name}=={pin}" for name, pin in wrong.items()]], check=True)
    importlib.invalidate_caches()
    # A package that was already imported keeps its old version until the session restarts.
    loaded = [name for name in wrong if name.replace("-", "_") in sys.modules]
    if loaded:
        raise RuntimeError(f"Installed the pinned versions of {', '.join(loaded)}. "
                           "Choose Runtime > Restart session, then Runtime > Run all.")
    wrong = {name: pin for name, pin in PINS.items() if installed_version(name) != pin}
if wrong:
    found = {name: installed_version(name) for name in wrong}
    raise RuntimeError(f"Versions differ from requirements.txt: {found}. "
                       "Run pip install -r requirements.txt, restart the kernel, then Run All.")

print(f"Python {sys.version.split()[0]} | {'Google Colab' if IN_COLAB else 'local'} | project: {ROOT}")
print(" | ".join(f"{name} {pin}" for name, pin in PINS.items()))

Python 3.13.18 | Google Colab | project: /content/data2010-steam-chinese
pandas 3.0.5 | numpy 2.5.3 | openpyxl 3.1.5 | matplotlib 3.11.2 | seaborn 0.13.2 | huggingface_hub 2.4.0 | pyarrow 26.0.1


In [44]:
# Import the libraries and name the project folders.
import ast                    # literal_eval: read "['English', 'French']" as a Python list
import csv                    # read the raw header row on its own
import gzip                   # unpack the release copy of an input file
import hashlib                # SHA-256 fingerprints of the input and output files
import re                     # regular expressions for text cleaning and name matching
import shutil                 # copy files without loading them into memory
import urllib.request         # download the release copy of an input file
import zipfile                # rewrite the xlsx files with fixed dates
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

RAW = ROOT / "data" / "raw"                           # the two input files (Git LFS)
PROCESSED_CSV = ROOT / "data" / "processed" / "csv"   # output tables for code
PROCESSED_XLSX = ROOT / "data" / "processed" / "xlsx" # same tables, to open and look at
PREFIX = "pcd_"                                       # file prefix of every output table
print("imports done; raw inputs in", RAW)

imports done; raw inputs in /content/data2010-steam-chinese/data/raw


# Step 1 · Clean the data and encode it → `pcd_games_clean`

**Strategy.** Load the two raw snapshots, clean both in exactly the same way (so a game
looks the same in both), remove the rows that cannot answer the question, and encode the
binned columns. The result for snapshot A is the cleaned dataset `pcd_games_clean`, one row
per game; the cleaned snapshot C is used again in Step 3.

| Sub-step | What | Why |
|---|---|---|
| 1.1 Load | check A and C against their SHA-256 fingerprints (fetch them if missing), keep 13 of 40 columns, English names | the same input bytes on every run |
| 1.2 Clean | fix types and data issues, parse list cells, add outcome, treatment, confounders and filter flags; encode the bins | below |
| 1.3 Filter | remove Chinese games, Early Access, < 10 reviews, < 1 year old, no genre/price/year | below |
| 1.4 Build | keep the columns of the cleaned dataset | |
| 1.5 Check | show how many games each filter removed and which Chinese games were removed | to check the rules by hand |

## 1.1 Load the two raw snapshots

Snapshot A (2026-06-20, the last version of `games.csv`; later commits only update a 1 GB
`games.json`) and snapshot C (2024-01-11), from the Hugging Face dataset
`FronkonGames/steam-games-dataset`, pinned by commit hash so the data never changes even
though the author keeps updating it.

**Same input bytes on every machine.** Each file is accepted only if its SHA-256 fingerprint
matches the one below; a single changed byte gives a different fingerprint. The notebook takes
the first copy that passes, in this order:

| Copy | When it is used |
| :--- | :--- |
| `data/raw/games_<a/c>.csv` | committed with Git LFS; present after a normal `git clone` |
| GitHub release `raw-snapshots-v1` (gzip) | Colab, or a clone made without Git LFS: there the two names hold 134-byte pointer files, which the fingerprint check rejects instead of failing later with shifted columns |
| Hugging Face, pinned commit | if the release cannot be reached |

**DATA ISSUE.** The header of snapshot A merges two names (`DiscountDLC count`) while every
row holds two values; without a fix every later column is shifted by one. The header is
repaired before reading, and the load stops with an error if the columns still look shifted.

In [45]:
# Fetch and verify the two snapshots, repair the header and read only the needed columns.

# Data source: two versions of the same file, pinned by commit hash.
HF_DATASET = "FronkonGames/steam-games-dataset"
HF_FILENAME = "games.csv"
REVISION = {
    "a": "52ba62abfa707a2fa31f4d147c2777a9d9fbd2da",   # snapshot A: last update of games.csv
    "c": "aefdeb1ff1a739b92ac5338682cc2610a0328505",   # snapshot C: January 2024 catalogue
}
SNAPSHOT_DATE_A = "2026-06-20"                          # day snapshot A was taken
SNAPSHOT_DATE_C = "2024-01-11"                          # day snapshot C was taken
# SHA-256 of each raw file; the same values are in SHA256SUMS.txt of the release.
SHA256 = {
    "a": "1b48008b01a799d82385d6e66ba0cb65dd477b6aa4b6c2ec8c443025ab6880ad",
    "c": "c755572b804a5c43f4f005aacef23cadfe92ae77d03daebf908d521c81285821",
}
RELEASE_URL = REPO_URL + "/releases/download/raw-snapshots-v1/games_{}.csv.gz"

# Raw column -> English snake_case name; only these 12 columns (+ discount) are read.
RAW_TO_CLEAN = {
    "AppID": "app_id",
    "Name": "name",
    "Release date": "release_date",
    "Price": "price",
    "Supported languages": "supported_languages",
    "Full audio languages": "full_audio_languages",
    "Positive": "positive",
    "Negative": "negative",
    "Developers": "developers",
    "Publishers": "publishers",
    "Genres": "genres",
    "Tags": "tags",
}
OPTIONAL_RAW_TO_CLEAN = {"Discount": "discount_pct"}   # only in snapshot A; "0" when missing


def sha256_of(path) -> str:
    """SHA-256 fingerprint of a file, read in 1 MB blocks."""
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def from_release(label: str, target) -> None:
    """Download the gzip copy of one snapshot from the GitHub release and unpack it."""
    packed = target.with_suffix(".csv.gz")
    urllib.request.urlretrieve(RELEASE_URL.format(label), packed)
    with gzip.open(packed, "rb") as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst)                       # unpack without loading it into memory
    packed.unlink()


def from_hugging_face(label: str, target) -> None:
    """Download games.csv of one snapshot from Hugging Face at its pinned commit."""
    from huggingface_hub import hf_hub_download
    cached = hf_hub_download(repo_id=HF_DATASET, filename=HF_FILENAME,
                             revision=REVISION[label], repo_type="dataset")
    shutil.copyfile(cached, target)                        # copy out of the download cache


def ensure_snapshot(label: str):
    """Path of data/raw/games_<label>.csv, fetched if missing or wrong, always with the right SHA-256."""
    RAW.mkdir(parents=True, exist_ok=True)
    target = RAW / f"games_{label}.csv"
    if target.exists() and sha256_of(target) == SHA256[label]:
        print(f"snapshot {label.upper()}: data/raw copy verified")
        return target
    for source, fetch in (("release raw-snapshots-v1", from_release),
                          ("Hugging Face", from_hugging_face)):
        try:
            fetch(label, target)
        except Exception as error:                          # no network, site down, ...
            print(f"snapshot {label.upper()}: {source} not reachable ({type(error).__name__})")
            continue
        if sha256_of(target) == SHA256[label]:
            print(f"snapshot {label.upper()}: downloaded from {source}, verified")
            return target
        print(f"snapshot {label.upper()}: copy from {source} has the wrong SHA-256")
    raise RuntimeError(f"snapshot {label.upper()}: no copy with SHA-256 {SHA256[label]} found")


def read_header(raw_path) -> list[str]:
    """Header row of a raw file, with the merged name split back into two (DATA ISSUE)."""
    with open(raw_path, newline="", encoding="utf-8") as f:
        header = next(csv.reader(f))                       # first row only
    fixed = []
    for name in header:
        # "DiscountDLC count" -> ["Discount", "DLC count"]; any other name stays as it is
        fixed += ["Discount", "DLC count"] if name == "DiscountDLC count" else [name]
    return fixed


def load_snapshot(raw_path) -> pd.DataFrame:
    """Read one raw snapshot: needed columns only, English names, every value as text."""
    names = read_header(raw_path)
    missing = [c for c in RAW_TO_CLEAN if c not in names]
    if missing:
        raise ValueError(f"{raw_path.name}: columns not found: {missing}")
    optional = [c for c in OPTIONAL_RAW_TO_CLEAN if c in names]
    raw = pd.read_csv(
        raw_path, names=names, header=None, skiprows=1,    # use the repaired header
        usecols=list(RAW_TO_CLEAN) + optional,             # read only these columns
        dtype=str, keep_default_na=False, na_values=[""],  # text; only "" counts as missing
        engine="c",
    )
    # Stop if columns are still shifted: language cells must look like "[...]" lists.
    langs = raw["Supported languages"].dropna()
    share_list_like = langs.str.startswith("[").mean() if len(langs) else 0
    if share_list_like < 0.9:
        raise ValueError(f"{raw_path.name}: columns look shifted; check read_header()")
    out = raw[list(RAW_TO_CLEAN)].rename(columns=RAW_TO_CLEAN)
    for raw_name, clean_name in OPTIONAL_RAW_TO_CLEAN.items():
        # copy the column when the snapshot has it, otherwise fill with "0"
        out[clean_name] = raw[raw_name] if raw_name in raw.columns else "0"
    return out

In [46]:
# Run: fetch (if needed), verify and read both snapshots.
raw_a = load_snapshot(ensure_snapshot("a"))
raw_c = load_snapshot(ensure_snapshot("c"))
print(f"snapshot A: {len(raw_a):,} games x {raw_a.shape[1]} columns")
print(f"snapshot C: {len(raw_c):,} games x {raw_c.shape[1]} columns")
raw_a.head(3)

snapshot A: data/raw copy verified
snapshot C: data/raw copy verified
snapshot A: 125,855 games x 13 columns
snapshot C: 85,103 games x 13 columns


,app_id,name,release_date,price,supported_languages,full_audio_languages,positive,negative,developers,publishers,genres,tags,discount_pct
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0.0,[],[],0,0,NaN,NaN,NaN,NaN,0
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",5.24,['English'],[],252,3,minori,MangaGamer,Adventure,"Adventure,Visual Novel,Anime,Cute",65
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",4.99,"['English', 'French', 'German', 'Russian']",[],21,3,Somer Games,8floor,Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",0


**Check.** Snapshot A has 125,855 games and snapshot C 85,103, each with 13 columns (the 12
named columns and the discount). Snapshot A's first rows show a sale discount (65) next to
the price, which Step 1.2 uses to recover the list price.

## 1.2 Clean each snapshot and add the analysis columns

No game is removed here (except rows without a usable id); every later filter only uses the
flags added here.

**Data issues fixed**
- **Sale prices.** Snapshot A was taken during a Steam sale (Hades: 7.49 = 24.99 at -70%).
  `price` becomes the list price, `sale price / (1 - discount / 100)`, so the price tier
  describes the game, not the sale. Snapshot C has no discount column; its prices are used
  as listed.
- **Junk in list cells.** Language and tag cells contain leftovers from the store pages
  (`English[b]*[/b]`, HTML tags, line breaks); they are removed before parsing.
- **Ids and dates.** Rows without an id or with a repeated id are dropped (first kept);
  unreadable release dates become missing and are dropped by the filters.

**Columns added**
- **Outcome:** `total_reviews = positive + negative` (reach).
- **Treatment:** supports Simplified Chinese. Traditional-only games are flagged: they are
  neither "with Simplified Chinese" nor "without Chinese".
- **Confounders** (characteristics that affect reach and differ between games with and
  without Chinese; Steps 2-3 hold them fixed):
  - `genre_group`: the first of the game's user tags found in the genre map. Steam orders
    tags by votes, so this is the genre players use most. Tags such as "Indie" are skipped:
    they say nothing about how much text a game has.
  - `price_tier`: list price in 5 tiers, a stand-in for budget and studio size.
  - `year_band`: release year in 4 bands (time on the store).
  - `n_other_langs`, `n_other_langs_bin`: languages other than English and Chinese, in 3 bins.
- **Chinese games** (flag; removed in 1.3). A game made in China has Chinese and Chinese
  players by default, so keeping it would mix "made for China" with "translated into
  Chinese". The data has no country column, so a game is Chinese if any rule matches:
  1. Chinese without English, in the text or in the voice-over. Chinese voice-over alone is
     not a rule: Cyberpunk 2077 and The Witcher 3 have it, and it would remove 4,233
     Western games.
  2. Every developer's name is in Chinese characters.
  3. Every developer's name contains a mainland Chinese city.
  4. Every developer is a known Chinese company (catches Black Myth: Wukong, which ships
     English). Rules 2-4 look at developers only, and need all of them to match, because
     publishers are often regional partners (bilibili publishes the Swedish Core Keeper in
     China) and big Western games list Chinese support studios (Ubisoft Shanghai on Far Cry).

**Encoding.** The three binned columns are saved as ordinal codes, the label's rank:

| Column | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| `price_tier` | free | < \$5 | \$5-10 | \$10-20 | > \$20 |
| `year_band` | <=2017 | 2018-2019 | 2020-2021 | 2022+ | |
| `n_other_langs_bin` | 0 | 1-4 | 5+ | | |

As text the labels sort in the wrong order ("10to20" before "5to10", "<=2017" after
"2022+"), which breaks tables and figure axes; codes sort correctly and can be used in
calculations that need an order. They are ranks, not amounts (tier 4 is not twice tier 2),
so they are not averaged or used as a plain number in a regression. `genre_group` is not
encoded: genres have no order, so a number would invent one.

In [47]:
# Rules used by the cleaning: languages, Chinese games, genre groups and bins.

# Languages: the treatment, and the language every Western game ships with.
ZH_SIMPLIFIED = "Simplified Chinese"
ZH_TRADITIONAL = "Traditional Chinese"
ENGLISH = "English"

# Chinese games: mainland cities and companies looked for in the developer names.
CN_CITIES = [
    "Beijing", "Shanghai", "Shenzhen", "Guangzhou", "Hangzhou", "Chengdu", "Nanjing",
    "Wuhan", "Xiamen", "Suzhou", "Chongqing", "Tianjin", "Changsha", "Fuzhou", "Hefei",
    "Jinan", "Qingdao", "Dalian", "Zhuhai", "Dongguan", "Xi'an",
]
CN_COMPANIES = [
    "Tencent", "NetEase", "miHoYo", "HoYoverse", "Game Science", "Perfect World",
    "X.D. Network", "XD Inc", "bilibili", "Gamera Games", "Hypergryph", "Lilith Games",
    "Seasun", "Kingsoft", "Papergames", "Paper Games", "Yostar", "Pathea", "NExT Studios",
    "TiMi Studio", "Leiting", "Giant Network", "Snail Games", "Shanda", "CMGE", "Zulong",
    "Hero Games", "Youthcat", "24 Entertainment",
]
# Legal suffixes that become separate items when a developer cell is split on commas.
COMPANY_SUFFIXES = ["inc", "ltd", "llc", "co", "corp", "limited", "gmbh", "plc", "sa", "srl", "co. ltd"]

# Genre groups: user tag -> group; the first matching tag of a game wins.
TAG_TO_GENRE = {
    "Visual Novel": "Visual Novel",
    "RPG": "RPG", "JRPG": "RPG", "CRPG": "RPG", "Action RPG": "RPG", "Party-Based RPG": "RPG",
    "Strategy": "Strategy", "Turn-Based Strategy": "Strategy", "RTS": "Strategy",
    "Grand Strategy": "Strategy", "4X": "Strategy", "Tower Defense": "Strategy",
    "Action": "Action", "Shooter": "Action", "FPS": "Action", "Platformer": "Action",
    "Hack and Slash": "Action", "Fighting": "Action",
    "Puzzle": "Puzzle", "Hidden Object": "Puzzle",
    "Casual": "Casual",
}

# Bins; each label is saved as its position 1, 2, 3, ... in the list.
PRICE_BINS = [-0.001, 0.0, 4.999, 9.999, 19.999, float("inf")]    # USD, list price
PRICE_LABELS = ["free", "lt5", "5to10", "10to20", "gt20"]           # codes 1-5
YEAR_BAND_BINS = [0, 2017, 2019, 2021, 2100]
YEAR_BAND_LABELS = ["<=2017", "2018-2019", "2020-2021", "2022+"]    # codes 1-4
OTHER_LANG_BINS = [-1, 0, 4, 10_000]
OTHER_LANG_LABELS = ["0", "1-4", "5+"]                              # codes 1-3

In [48]:
# Helper functions: clean list-like text cells, pick the genre, split developer names.

# DATA ISSUE: store-page leftovers in list cells: BBCode tags, HTML tags/entities, "*", line breaks
_TAG_JUNK = re.compile(r"\[/?[a-zA-Z]+\]|<[^>]+>|&amp;|&lt;|&gt;|\*|\r|\n|#lang_\w+")


def _word_regex(words: list[str]) -> re.Pattern:
    """Case-insensitive pattern matching any of `words` as a whole word."""
    return re.compile(r"\b(?:" + "|".join(re.escape(w) for w in words) + r")\b", re.IGNORECASE)


_CN_CITY = _word_regex(CN_CITIES)           # e.g. "Shanghai ... Co., Ltd."
_CN_COMPANY = _word_regex(CN_COMPANIES)     # e.g. "Game Science"
_CJK = re.compile(r"[一-鿿]")       # any Chinese character


def parse_list(value) -> list[str]:
    """List-like cell -> clean list of unique strings ("['A', 'B']", "A,B", junk, missing)."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []                                          # missing cell
    text = _TAG_JUNK.sub("", str(value)).strip()           # remove the junk first
    if not text or text in {"[]", "nan"}:
        return []
    items = None
    if text.startswith("["):
        try:
            items = ast.literal_eval(text)                 # "['A', 'B']" -> ['A', 'B']
        except (ValueError, SyntaxError):
            text = text.strip("[]")                        # broken list: fall back to commas
    if items is None:
        items = text.split(",")                            # "A,B" -> ['A', 'B']
    cleaned = [str(i).strip().strip("'\"").strip() for i in items]
    seen, out = set(), []
    for item in cleaned:                                   # drop empties and duplicates,
        if item and item not in seen:                      # keep the original order
            seen.add(item)
            out.append(item)
    return out


def first_genre(tags: list[str]) -> str | None:
    """Group of the first tag that appears in TAG_TO_GENRE; None if no tag matches."""
    for tag in tags:
        if tag in TAG_TO_GENRE:
            return TAG_TO_GENRE[tag]
    return None


def split_developers(text: str) -> list[str]:
    """"A, B, Inc." -> ["A", "B"]: split on commas and drop legal suffixes."""
    parts = [p.strip().strip(".").strip() for p in str(text).split(",")]
    suffixes = set(COMPANY_SUFFIXES)
    return [p for p in parts if p and p.lower().rstrip(".") not in suffixes]


def all_match(developers: list[str], pattern: re.Pattern) -> bool:
    """True if there is at least one developer and every developer matches `pattern`."""
    return bool(developers) and all(pattern.search(d) for d in developers)

In [49]:
# Clean one snapshot and add every analysis column (outcome, treatment, confounders, flags).

def to_code(values: pd.Series, bins: list) -> pd.Series:
    """Bin numbers and return the bin's rank 1, 2, 3, ... (missing stays missing)."""
    # labels=False gives the bin position 0, 1, 2, ...; +1 makes it 1, 2, 3, ...
    return (pd.cut(values.astype("float"), bins, labels=False) + 1).astype("Int64")


def add_features(trim: pd.DataFrame, snapshot_date: str) -> pd.DataFrame:
    """Cleaned copy of one snapshot with all derived columns; only rows without an id are dropped."""
    df = trim.copy()

    # --- 1. types
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")        # bad id -> NaN
    df = df.dropna(subset=["app_id"]).drop_duplicates("app_id")        # DATA ISSUE: no / repeated id
    df["app_id"] = df["app_id"].astype("int64")
    for col in ("positive", "negative"):
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")
    # DATA ISSUE: sale prices in A -> list price = price / (1 - discount / 100)
    df["price_at_snapshot"] = pd.to_numeric(df["price"], errors="coerce")
    discount = df["discount_pct"] if "discount_pct" in df else pd.Series(0, index=df.index)
    df["discount_pct"] = pd.to_numeric(discount, errors="coerce").fillna(0).clip(0, 99)
    df["price"] = (df["price_at_snapshot"] / (1 - df["discount_pct"] / 100)).round(2)
    # DATA ISSUE: dates come as "Aug 1, 2023" or "Aug 2023"; unreadable -> NaT (missing)
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
    df["release_year"] = df["release_date"].dt.year.astype("Int64")
    df["days_since_release"] = (pd.Timestamp(snapshot_date) - df["release_date"]).dt.days

    # --- 2. list-like text cells -> Python lists
    langs = df["supported_languages"].map(parse_list)
    audio = df["full_audio_languages"].map(parse_list)
    tags = df["tags"].map(parse_list)
    genres = df["genres"].map(parse_list)

    # --- 3. outcome
    df["total_reviews"] = df["positive"] + df["negative"]

    # --- 4. languages: treatment, other-languages confounder, text lists for Step 3
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["has_zh_simplified"] = langs.map(lambda x: ZH_SIMPLIFIED in x)
    df["has_zh_traditional"] = langs.map(lambda x: ZH_TRADITIONAL in x)
    df["has_english"] = langs.map(lambda x: ENGLISH in x)
    df["n_other_langs"] = langs.map(lambda x: len(set(x) - chinese - {ENGLISH}))   # set difference
    df["n_other_langs_bin"] = to_code(df["n_other_langs"], OTHER_LANG_BINS)      # codes 1-3
    df["all_langs"] = langs.map(lambda x: "|".join(sorted(x)))                   # "English|French|..."

    # --- 5. Chinese games: one True/False column per rule, then "any rule matched"
    has_zh_audio = audio.map(lambda x: bool(set(x) & chinese))
    has_en_audio = audio.map(lambda x: ENGLISH in x)
    any_zh = df["has_zh_simplified"] | df["has_zh_traditional"]
    developers = df["developers"].fillna("").astype(str)
    publishers = df["publishers"].fillna("").astype(str)
    dev_lists = developers.map(split_developers)
    rules = {
        "chinese_without_english": (any_zh & ~df["has_english"]) | (has_zh_audio & ~has_en_audio),
        "developer_name_in_chinese": dev_lists.map(lambda d: all_match(d, _CJK)),
        "developer_in_chinese_city": dev_lists.map(lambda d: all_match(d, _CN_CITY)),
        "developer_known_chinese_company": dev_lists.map(lambda d: all_match(d, _CN_COMPANY)),
    }
    # (Python's re via .map, not .str.contains: with pyarrow installed pandas uses Arrow's
    # regex engine, which rejects some of these patterns.)
    rule_frame = pd.DataFrame(rules)
    df["is_cn_game"] = rule_frame.any(axis=1)                          # True if any rule matched
    # names of the matched rules joined by ";", for the check in 1.5
    df["cn_rules"] = rule_frame.apply(lambda r: ";".join(r.index[r.to_numpy()]), axis=1)
    df["makers"] = developers + " | " + publishers
    df["has_zh_audio"] = has_zh_audio

    # --- 6. other filter flags
    df["is_early_access"] = genres.map(lambda x: "Early Access" in x)
    df["is_traditional_only"] = df["has_zh_traditional"] & ~df["has_zh_simplified"]

    # --- 7. confounders: genre group and the encoded bins
    df["genre_group"] = tags.map(first_genre)
    df["price_tier"] = to_code(df["price"], PRICE_BINS)                # codes 1-5
    df["year_band"] = to_code(df["release_year"], YEAR_BAND_BINS)      # codes 1-4
    return df

In [50]:
# Run: clean both snapshots the same way.
feat_a = add_features(raw_a, SNAPSHOT_DATE_A)
feat_c = add_features(raw_c, SNAPSHOT_DATE_C)
print(f"A: {len(feat_a):,} games, C: {len(feat_c):,} games")

A: 125,855 games, C: 85,103 games


**Check.** No game is removed yet: A still has 125,855 games and C 85,103.

## 1.3 Filter the rows

Applied in this order; each step is logged in a funnel (shown in 1.5).

| Filter | Why |
|---|---|
| drop Chinese games (any of the 4 rules) | made for China, not translated into Chinese |
| drop Early Access | unfinished games; their reviews behave differently |
| keep >= 10 reviews | below that, reach is noise; not higher (e.g. 500), because filtering hard on the outcome itself would shrink the gap being measured |
| keep games released >= 1 year before the snapshot | every game had time to collect reviews |
| keep games with a genre group, price and release year | needed to place the game in a comparison group |
| `pcd_games_clean` only: drop Traditional-Chinese-only games | neither "with Simplified Chinese" nor "without Chinese" |

In [51]:
# Functions to apply named filters in order and record how many games survive each one.

MIN_REVIEWS = 10              # below that, reach is noise (see the table above)
MIN_AGE_DAYS = 365            # every game had a year to collect reviews

def apply_filters(df: pd.DataFrame, steps: list[tuple[str, pd.Series]], table: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Keep rows passing every (name, mask) filter; return them and the funnel table."""
    rows = [{"table": table, "step": "start", "games_left": len(df)}]
    keep = pd.Series(True, index=df.index)
    for name, mask in steps:
        keep &= mask.reindex(df.index).fillna(False)       # missing value = fails the filter
        rows.append({"table": table, "step": name, "games_left": int(keep.sum())})
    return df[keep].copy(), pd.DataFrame(rows)


def base_filters(df: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    """Filters used for both snapshots, as (name, True = keep) pairs, in order."""
    return [
        ("drop Chinese games (any of 4 rules)", ~df["is_cn_game"]),
        ("drop Early Access", ~df["is_early_access"]),
        (f"keep >= {MIN_REVIEWS} reviews", df["total_reviews"] >= MIN_REVIEWS),
        ("keep released >= 1 year before snapshot", df["days_since_release"] >= MIN_AGE_DAYS),
        ("keep games with a genre group, price and year",
         df["genre_group"].notna() & df["price"].notna() & df["release_year"].notna()),
    ]

## 1.4 Build the cleaned dataset `pcd_games_clean`

One row per game of snapshot A that passed every filter: identifiers, confounders (raw
value and code), the treatment `has_zh` (1 = Simplified Chinese, 0 = no Chinese at all) and
the outcome. `log_total_reviews` is added because review counts are very skewed (a few hits
have millions), so figures read better on a log scale.

In [52]:
# Select the rows and columns of the cleaned dataset (snapshot A).

def build_games_clean(feat_a: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Cleaned snapshot A: one row per game after all filters, and its funnel."""
    steps = base_filters(feat_a) + [("drop Traditional-Chinese-only", ~feat_a["is_traditional_only"])]
    df, funnel = apply_filters(feat_a, steps, "games_clean")
    df["has_zh"] = df["has_zh_simplified"].astype(int)               # True/False -> 1/0
    df["log_total_reviews"] = np.log10(df["total_reviews"]).round(6) # rounded: same bytes every run
    for col in ["price_tier", "year_band", "n_other_langs_bin"]:
        df[col] = df[col].astype(int)                                # no missing left after filters
    cols = [
        "app_id", "name", "genre_group", "release_year", "year_band", "price", "price_tier",
        "n_other_langs", "n_other_langs_bin", "has_zh", "total_reviews", "log_total_reviews",
    ]
    return df[cols].sort_values("app_id").reset_index(drop=True), funnel

In [53]:
# Run: build the cleaned dataset.
games_clean, funnel_a = build_games_clean(feat_a)
print(f"games_clean: {len(games_clean):,} games, {games_clean['has_zh'].sum():,} with Simplified Chinese")
games_clean.head()

games_clean: 44,765 games, 14,078 with Simplified Chinese


,app_id,name,genre_group,release_year,year_band,price,price_tier,n_other_langs,n_other_langs_bin,has_zh,total_reviews,log_total_reviews
0,10,Counter-Strike,Action,2000,1,9.95,3,5,3,1,250245,5.398365
1,20,Team Fortress Classic,Action,1999,1,4.96,2,6,3,1,8738,3.941412
2,30,Day of Defeat,Action,2003,1,4.96,2,4,2,0,7102,3.851381
3,40,Deathmatch Classic,Action,2001,1,4.96,2,6,3,1,3163,3.500099
4,50,Half-Life: Opposing Force,Action,1999,1,4.96,2,3,2,0,25561,4.407578


**Check.** `games_clean` has 44,765 games, 14,078 of them with Simplified Chinese.

## 1.5 Check what the cleaning removed

The funnel shows how many games are left after each filter; no single step should remove
most games. The list of removed Chinese games (biggest first) is there to spot a wrongly
removed Western game by eye. Shown only, not saved.

In [54]:
# Show the funnel of snapshot A and the 20 biggest Chinese games removed.
cn_cols = ["app_id", "name", "makers", "cn_rules", "has_english", "has_zh_audio", "total_reviews"]
cn_excluded = (
    feat_a[feat_a["is_cn_game"] & (feat_a["total_reviews"] >= MIN_REVIEWS)][cn_cols]  # would pass the review filter
    .sort_values("total_reviews", ascending=False)                                   # biggest first
    .reset_index(drop=True)
)
print(f"{len(cn_excluded):,} Chinese games removed (>= {MIN_REVIEWS} reviews)")
display(funnel_a)
cn_excluded.head(20)

2,862 Chinese games removed (>= 10 reviews)


,table,step,games_left
0,games_clean,start,125855
1,games_clean,drop Chinese games (any of 4 rules),119738
2,games_clean,drop Early Access,108702
3,games_clean,keep >= 10 reviews,49372
4,games_clean,keep released >= 1 year before snapshot,49052
5,games_clean,"keep games with a genre group, price and year",45431
6,games_clean,drop Traditional-Chinese-only,44765


,app_id,name,makers,cn_rules,has_english,has_zh_audio,total_reviews
0,2358720,Black Myth: Wukong,Game Science | Game Science,developer_known_chinese_company,True,True,1150098
1,1203220,NARAKA: BLADEPOINT,24 Entertainment | NetEase Games Global,developer_known_chinese_company,True,True,324927
2,2767030,Marvel Rivals,NetEase Games | NetEase Games,developer_known_chinese_company,True,True,287569
3,1468810,鬼谷八荒 Tale of Immortal,"鬼谷工作室 | 鬼谷工作室,Lightning Games",developer_name_in_chinese,True,False,227104
4,1366540,Dyson Sphere Program,Youthcat Studio | Gamirror Games,developer_known_chinese_company,True,False,85088
5,838350,The Scroll Of Taiwu,ConchShip Games | ConchShip Games,chinese_without_english,False,False,60637
6,2593370,The Hungry Lamb: Traveling in the Late Ming Dy...,零创游戏(ZerocreationGame) | 零创游戏(ZerocreationGame...,chinese_without_english;developer_name_in_chinese,True,True,52780
7,994280,Gujian3(古剑奇谭三),"上海烛龙信息科技有限公司,Aurogon Shanghai | 北京网元圣唐娱乐科技有限公司",chinese_without_english,True,True,47845
8,2322560,Love Is All Around,intiny | intiny,chinese_without_english,True,True,45414
9,666140,My Time at Portia,Pathea Games | Focus Entertainment,developer_known_chinese_company,True,True,43270


**Why?** The largest drop in the funnel is the review filter (108,702 → 49,372 games): most
games on Steam have fewer than 10 reviews. 2,862 Chinese games with at least 10 reviews are
removed; the top of the list (Black Myth: Wukong, NARAKA: BLADEPOINT, Marvel Rivals) is made in
China, which is what the rules are meant to catch.

# Step 2 · SQ1 groups → `pcd_sq1_groups`

**Question.** How much larger is the reach of games that support Simplified Chinese, and in
which genres?

**Strategy: compare only similar games.** Comparing all games with and without Chinese
directly would mix the effect of Chinese with that of the confounders (bigger-budget games
are both more often translated and more reviewed). So every game of `pcd_games_clean` is put
in a **comparison group** of similar games, and games with and without Chinese are compared
**only inside the same group**: there the confounders are the same on both sides, so the
remaining gap is not explained by them. This is why the confounders need no other treatment
in the analysis.

**SQ1 group = genre x price tier x release-year band x number of other languages**

| Confounder | Why it is in the group |
|---|---|
| Genre | genres differ in audience size and in how much text a game has |
| Price tier | budget and studio size drive both translation and reach |
| Release-year band | older games have had more time to collect reviews; bands rather than single years, because single years leave only 44% of games in usable groups (76% with bands) |
| Other languages | the strongest: games with Chinese have a median of 7 other languages, games without it 0 (69% vs 9% have 5+); without it SQ1 would compare "localised in many languages" with "not localised" rather than measure Chinese |

**The table `pcd_sq1_groups`**, one row per group: number of games and **median** reviews
on each side (medians, because review counts are very skewed), and the gap
`pct_diff = median with Chinese / median without - 1`. A group is `eligible` only with >= 10
games on each side; a median of fewer games moves too much with one game. Result: 360
groups, 178 eligible, covering 76% of games (87% of games with Chinese).

In [55]:
# Build the SQ1 comparison table: one row per group, games with vs without Chinese.

MIN_GAMES_PER_SIDE = 10       # a group is eligible only with this many games per side (also Step 3)

# code -> label, to write readable group names ("Action | <=2017 | lt5 | 5+")
PRICE_LABEL = dict(enumerate(PRICE_LABELS, start=1))
YEAR_BAND_LABEL = dict(enumerate(YEAR_BAND_LABELS, start=1))
OTHER_LANGS_LABEL = dict(enumerate(OTHER_LANG_LABELS, start=1))


def add_sq1_group(games: pd.DataFrame) -> pd.DataFrame:
    """Copy of games with group_id = genre | year band | price tier | other languages."""
    return games.assign(group_id=(
        games["genre_group"] + " | " + games["year_band"].map(YEAR_BAND_LABEL) + " | "
        + games["price_tier"].map(PRICE_LABEL) + " | " + games["n_other_langs_bin"].map(OTHER_LANGS_LABEL)
    ))


def build_sq1_groups(games: pd.DataFrame) -> pd.DataFrame:
    """One row per group: games and median reviews with / without Chinese, gap, eligible."""
    keys = ["group_id", "genre_group", "year_band", "price_tier", "n_other_langs_bin"]
    wide = (
        games.groupby(keys + ["has_zh"])["total_reviews"]
        .agg(n="size", median="median")                    # count and median per group and side
        .unstack("has_zh")                                 # sides become columns: (stat, 0/1)
    )
    out = pd.DataFrame({
        "n_zh": wide[("n", 1)],
        "n_no_zh": wide[("n", 0)],
        "median_reviews_zh": wide[("median", 1)],
        "median_reviews_no_zh": wide[("median", 0)],
    }).reset_index()
    out[["n_zh", "n_no_zh"]] = out[["n_zh", "n_no_zh"]].fillna(0).astype(int)  # empty side -> 0
    out["pct_diff"] = (out["median_reviews_zh"] / out["median_reviews_no_zh"] - 1).round(6)
    out["eligible"] = (out["n_zh"] >= MIN_GAMES_PER_SIDE) & (out["n_no_zh"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)   # genre, then codes: old -> new, cheap -> expensive

In [56]:
# Run: build the SQ1 group table.
sq1_groups = build_sq1_groups(add_sq1_group(games_clean))
print(f"sq1_groups: {len(sq1_groups)} groups, {sq1_groups['eligible'].sum()} eligible")
sq1_groups.head()

sq1_groups: 360 groups, 178 eligible


,group_id,genre_group,year_band,price_tier,n_other_langs_bin,n_zh,n_no_zh,median_reviews_zh,median_reviews_no_zh,pct_diff,eligible
0,Action | <=2017 | free | 0,Action,1,1,1,6,348,54.0,103.5,-0.478261,False
1,Action | <=2017 | free | 1-4,Action,1,1,2,11,99,45.0,310.0,-0.854839,True
2,Action | <=2017 | free | 5+,Action,1,1,3,77,52,8617.0,1328.0,5.488705,True
3,Action | <=2017 | lt5 | 0,Action,1,2,1,19,1344,76.0,64.5,0.178295,True
4,Action | <=2017 | lt5 | 1-4,Action,1,2,2,42,395,65.0,114.0,-0.429825,True


**Check.** 360 groups, 178 of them eligible.

# Step 3 · SQ2 groups → `pcd_sq2_groups`

**Question.** Does adding Simplified Chinese after launch still increase reach, and in
which genres?

**Strategy: follow the same games over time.** Take the games that had no Chinese in
January 2024 (snapshot C) and are still on Steam in June 2026 (snapshot A), and measure how
much their reviews grew. Games that added Chinese in between (**treated**) are compared with
similar games that added nothing (**control**). The control arm is needed because every game
gains reviews over time anyway; the uplift is the extra growth of the treated games.

**3.1 The panel** (one row per game, the input of the groups; not saved):
- start from cleaned snapshot C with the same filters as Step 1 (so the starting point is
  measured the same way), keep games with no Chinese at all in C and still present in A;
- game characteristics are taken from C, because what a game looked like **before** adding
  Chinese is what is held fixed;
- drop games with 0 reviews in A (delisted or missing data: a growth of -100% is not real);
- outcome `growth = (reviews in A - reviews in C) / reviews in C`;
- `arm`: `treated` = added Simplified Chinese; `control` = added neither Chinese nor any
  major language (German, French, Spanish, Japanese, Russian, Korean, Portuguese-Brazil),
  because a game that added German and French at the same time had a localisation push of
  its own and is not a clean "did nothing"; `excluded` = added another major language only.

**3.2 SQ2 group = genre x release-year band.** Only 509 games added Chinese, so the group
can hold only the confounders that matter most for **growth**:

| Variable | In the group? | Why |
|---|---|---|
| Genre | yes | needed to answer per genre |
| Release-year band | yes | the strongest: among control games, median growth is +12% for games from <=2017 and +77% for 2022+, and treated games are about 2 years newer; without it the effect of Chinese would be overstated |
| Price tier | no | once the year band is fixed, growth is about the same in every tier except > \$20; genre x price x year leaves only 17 usable groups, 54% of treated games and 3 genres (no RPG, Strategy or Visual Novel) |
| Other languages | no | treated and control games have almost the same mix (0 / 1-4 / 5+: 62/25/13% vs 64/26/10%) |

Result: 24 groups, 16 eligible, covering 90% of treated games and all 6 genres.

**The table `pcd_sq2_groups`**, built as in Step 2 but with growth: number of treated and
control games, their median growth, and
`uplift_pts = median growth treated - median growth control` (0.1 = +10 growth points);
`eligible` with >= 10 games on each side.

In [57]:
# Build the SQ2 panel: games without Chinese in C, still in A, with their growth and arm.

MAJOR_LANGS = [               # a control game must not have added any of these
    "German", "French", "Spanish - Spain", "Japanese", "Russian",
    "Korean", "Portuguese - Brazil",
]

def build_sq2_panel(feat_a: pd.DataFrame, feat_c: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One row per game of C (no Chinese, filtered as in Step 1) also present in A; and its funnel."""
    steps = base_filters(feat_c) + [
        ("keep games with no Chinese (Simplified or Traditional) in C",
         ~(feat_c["has_zh_simplified"] | feat_c["has_zh_traditional"])),
        ("keep games also present in A", feat_c["app_id"].isin(feat_a["app_id"])),
    ]
    c, funnel = apply_filters(feat_c, steps, "sq2_panel")
    c = c.set_index("app_id")
    a = feat_a.set_index("app_id").loc[c.index]            # same games, same order, from A

    p = pd.DataFrame(index=c.index)
    for col in ["name", "genre_group", "release_year", "year_band", "price_tier"]:
        p[col] = c[col]                                    # characteristics BEFORE: from C
    p["price_c"] = c["price"]
    p["reviews_c"] = c["total_reviews"]
    p["reviews_a"] = a["total_reviews"]
    p = p[p["reviews_a"] > 0].copy()                       # drop delisted games (0 reviews in A)
    a, c = a.loc[p.index], c.loc[p.index]
    p["growth"] = ((p["reviews_a"] - p["reviews_c"]) / p["reviews_c"]).round(6)

    # languages in A but not in C = added after January 2024 (set difference per game)
    new_langs = [
        set(la.split("|")) - set(lc.split("|")) - {""}
        for la, lc in zip(a["all_langs"], c["all_langs"])
    ]
    added_zh = a["has_zh_simplified"].to_numpy()           # had no Chinese in C, has it in A
    added_major = np.array([bool(n & set(MAJOR_LANGS)) for n in new_langs])
    # np.select: first condition that is True gives the label, otherwise the default
    p["arm"] = np.select([added_zh, ~added_zh & ~added_major], ["treated", "control"], default="excluded")
    for col in ["year_band", "price_tier"]:
        p[col] = p[col].astype(int)                        # no missing left after filters
    funnel = pd.concat([funnel, pd.DataFrame([
        {"table": "sq2_panel", "step": "keep games with reviews in A (not delisted)", "games_left": len(p)},
        {"table": "sq2_panel", "step": "arm = treated", "games_left": int((p["arm"] == "treated").sum())},
        {"table": "sq2_panel", "step": "arm = control", "games_left": int((p["arm"] == "control").sum())},
    ])], ignore_index=True)
    cols = ["name", "genre_group", "release_year", "year_band", "price_c", "price_tier",
            "reviews_c", "reviews_a", "growth", "arm"]
    return p[cols].reset_index().sort_values("app_id").reset_index(drop=True), funnel

In [58]:
# Run: build the panel and show its funnel.
sq2_panel, funnel_c = build_sq2_panel(feat_a, feat_c)
print(f"sq2_panel: {len(sq2_panel):,} games")
print(sq2_panel["arm"].value_counts().to_string())
display(funnel_c)

sq2_panel: 21,555 games
arm
control     20805
treated       509
excluded      241


,table,step,games_left
0,sq2_panel,start,85103
1,sq2_panel,drop Chinese games (any of 4 rules),81064
2,sq2_panel,drop Early Access,71227
3,sq2_panel,keep >= 10 reviews,36152
4,sq2_panel,keep released >= 1 year before snapshot,34153
5,sq2_panel,"keep games with a genre group, price and year",31303
6,sq2_panel,keep games with no Chinese (Simplified or Trad...,22943
7,sq2_panel,keep games also present in A,21595
8,sq2_panel,keep games with reviews in A (not delisted),21555
9,sq2_panel,arm = treated,509


**Check.** The panel has 21,555 games: 509 treated, 20,805 control and 241 excluded.

In [59]:
# Build the SQ2 comparison table: one row per group, treated vs control growth.

def add_sq2_group(panel: pd.DataFrame) -> pd.DataFrame:
    """Copy of the panel with group_id_sq2 = genre | year band."""
    return panel.assign(group_id_sq2=panel["genre_group"] + " | " + panel["year_band"].map(YEAR_BAND_LABEL))


def build_sq2_groups(panel: pd.DataFrame) -> pd.DataFrame:
    """One row per group: treated and control games, their median growth, uplift, eligible."""
    keys = ["group_id_sq2", "genre_group", "year_band"]
    used = panel[panel["arm"].isin(["treated", "control"])]          # "excluded" is not compared
    wide = used.groupby(keys + ["arm"])["growth"].agg(n="size", median="median").unstack("arm")
    out = pd.DataFrame({
        "n_treated": wide[("n", "treated")],
        "n_control": wide[("n", "control")],
        "median_growth_treated": wide[("median", "treated")].round(6),
        "median_growth_control": wide[("median", "control")].round(6),
    }).reset_index()
    out[["n_treated", "n_control"]] = out[["n_treated", "n_control"]].fillna(0).astype(int)
    out["uplift_pts"] = (out["median_growth_treated"] - out["median_growth_control"]).round(6)
    out["eligible"] = (out["n_treated"] >= MIN_GAMES_PER_SIDE) & (out["n_control"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)          # genre, then old -> new

In [60]:
# Run: build the SQ2 group table.
sq2_groups = build_sq2_groups(add_sq2_group(sq2_panel))
print(f"sq2_groups: {len(sq2_groups)} groups, {sq2_groups['eligible'].sum()} eligible")
sq2_groups

sq2_groups: 24 groups, 16 eligible


,group_id_sq2,genre_group,year_band,n_treated,n_control,median_growth_treated,median_growth_control,uplift_pts,eligible
0,Action | <=2017,Action,1,41,3891,0.171821,0.105263,0.066558,True
1,Action | 2018-2019,Action,2,31,1960,0.175252,0.147244,0.028008,True
2,Action | 2020-2021,Action,3,49,1792,0.285714,0.255392,0.030322,True
3,Action | 2022+,Action,4,44,444,1.347716,0.700000,0.647716,True
4,Casual | <=2017,Casual,1,24,2313,0.134079,0.142857,-0.008778,True
5,Casual | 2018-2019,Casual,2,44,1630,0.142857,0.210526,-0.067669,True
6,Casual | 2020-2021,Casual,3,41,1277,0.250000,0.266667,-0.016667,True
7,Casual | 2022+,Casual,4,27,278,1.261905,0.666667,0.595238,True
8,Puzzle | <=2017,Puzzle,1,16,644,0.229809,0.147938,0.081871,True
9,Puzzle | 2018-2019,Puzzle,2,5,275,0.495974,0.218750,0.277224,False


**Check.** 24 groups, 16 of them eligible.

# Save and check

The three tables are written whole as csv (for code) and xlsx (to view). An xlsx file is a zip
archive that records the time it was written, inside the file and on every zip entry; both
are set to a fixed date, so its content depends only on the table. The last lines printed are
fingerprints of the saved tables: the same values on every machine (for an xlsx, of its
unzipped content, since zip compression differs between zlib builds). The checks then stop the
notebook with an error if anything is wrong, including any of the numbers in the **Check**
notes above.

In [61]:
# Write the three output tables as csv and xlsx, prefix pcd_.
tables = {
    "games_clean": games_clean,   # Step 1: cleaned data, one row per game
    "sq1_groups": sq1_groups,     # Step 2: one row per SQ1 group
    "sq2_groups": sq2_groups,     # Step 3: one row per SQ2 group
}
FIXED_DATE = datetime(2026, 1, 1)                     # written into every xlsx instead of "now"


def write_xlsx(table: pd.DataFrame, path, sheet: str) -> None:
    """Write an xlsx whose bytes depend only on the table: fixed dates inside and on the zip entries."""
    draft = path.with_name(path.stem + "_draft.xlsx")
    with pd.ExcelWriter(draft, engine="openpyxl") as writer:
        table.to_excel(writer, index=False, sheet_name=sheet)
    stamp = FIXED_DATE.strftime("%Y-%m-%dT%H:%M:%SZ").encode()
    with zipfile.ZipFile(draft) as src, zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as dst:
        for item in src.infolist():
            data = src.read(item.filename)
            if item.filename == "docProps/core.xml":       # openpyxl writes the save time here
                data = re.sub(rb"(<dcterms:(?:created|modified)[^>]*>)[^<]*", rb"\g<1>" + stamp, data)
            entry = zipfile.ZipInfo(item.filename, date_time=FIXED_DATE.timetuple()[:6])
            entry.compress_type = zipfile.ZIP_DEFLATED
            dst.writestr(entry, data)                      # same entries, fixed timestamps
    draft.unlink()


PROCESSED_CSV.mkdir(parents=True, exist_ok=True)
PROCESSED_XLSX.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    table.to_csv(PROCESSED_CSV / f"{PREFIX}{name}.csv", index=False)
    write_xlsx(table, PROCESSED_XLSX / f"{PREFIX}{name}.xlsx", sheet=name[:31])   # sheet names max 31 chars
    print(f"{PREFIX}{name}: {len(table):,} rows x {table.shape[1]} columns")

pcd_games_clean: 44,765 rows x 12 columns
pcd_sq1_groups: 360 rows x 11 columns
pcd_sq2_groups: 24 rows x 9 columns


In [62]:
# Stop with an error if any table is wrong.

# structure: unique ids, codes in range, English snake_case column names
assert games_clean["app_id"].is_unique and sq2_panel["app_id"].is_unique
assert set(games_clean["has_zh"]) <= {0, 1}
assert set(sq2_panel["arm"]) <= {"treated", "control", "excluded"}
assert games_clean["price_tier"].between(1, 5).all() and games_clean["year_band"].between(1, 4).all()
assert games_clean["n_other_langs_bin"].between(1, 3).all()
for name, table in tables.items():
    assert all(re.fullmatch(r"[a-z0-9_]+", c) for c in table.columns), name

# saved files: the xlsx holds the same table as the csv
for name, table in tables.items():
    back = pd.read_excel(PROCESSED_XLSX / f"{PREFIX}{name}.xlsx")
    assert back.shape == table.shape and list(back.columns) == list(table.columns), name

# known games: data issues and rules behave as intended
by_name = games_clean.set_index("name")
assert by_name.loc["Hades", "price"] > 20                             # list price, not the 7.49 sale price
assert by_name.loc["Hades", "price_tier"] == PRICE_LABELS.index("gt20") + 1
for name in ["Cyberpunk 2077", "Far Cry® 5", "Core Keeper", "Hollow Knight"]:
    assert name in by_name.index, f"{name} should be kept"            # Western games kept
for name in ["Black Myth: Wukong", "Marvel Rivals", "Dyson Sphere Program"]:
    assert name in set(cn_excluded["name"]), f"{name} should be removed"  # Chinese games removed
assert sq2_panel["growth"].min() > -1                                 # no delisted game left

# group tables: every game in exactly one group, eligible groups really have >= 10 games per side
assert sq1_groups[["n_zh", "n_no_zh"]].to_numpy().sum() == len(games_clean)
assert sq2_groups[["n_treated", "n_control"]].to_numpy().sum() == (sq2_panel["arm"] != "excluded").sum()
assert (sq1_groups.loc[sq1_groups["eligible"], ["n_zh", "n_no_zh"]] >= MIN_GAMES_PER_SIDE).all().all()
assert (sq2_groups.loc[sq2_groups["eligible"], ["n_treated", "n_control"]] >= MIN_GAMES_PER_SIDE).all().all()
# the run reproduces the numbers given in the Check notes and in the README
assert (len(raw_a), len(raw_c)) == (125_855, 85_103)
assert (len(games_clean), int(games_clean["has_zh"].sum())) == (44_765, 14_078)
assert len(cn_excluded) == 2_862
assert (len(sq1_groups), int(sq1_groups["eligible"].sum())) == (360, 178)
assert sq2_panel["arm"].value_counts().to_dict() == {"control": 20_805, "treated": 509, "excluded": 241}
assert (len(sq2_groups), int(sq2_groups["eligible"].sum())) == (24, 16)
print("all checks passed")

# Fingerprints of the saved tables: two runs on any machine print the same values.
def content_fingerprint(path) -> str:
    """SHA-256 of a csv file, or of the unzipped entries of an xlsx file."""
    if path.suffix != ".xlsx":
        return sha256_of(path)
    digest = hashlib.sha256()
    with zipfile.ZipFile(path) as archive:
        for name in sorted(archive.namelist()):
            digest.update(name.encode() + archive.read(name))
    return digest.hexdigest()


for path in sorted([*PROCESSED_CSV.glob(f"{PREFIX}*.csv"), *PROCESSED_XLSX.glob(f"{PREFIX}*.xlsx")]):
    print(f"{content_fingerprint(path)[:16]}  {path.relative_to(ROOT)}")

all checks passed
1adb22a86bcb059b  data/processed/csv/pcd_games_clean.csv
dd8f71fc383f212d  data/processed/csv/pcd_sq1_groups.csv
c0beebd39800eb0b  data/processed/csv/pcd_sq2_groups.csv
042e992328f14d54  data/processed/xlsx/pcd_games_clean.xlsx
2ee450d46b8384a1  data/processed/xlsx/pcd_sq1_groups.xlsx
001262dccd5d0254  data/processed/xlsx/pcd_sq2_groups.xlsx


# Part B · Analysis: figures and insights

Steps 1-3 did the data wrangling; Part B only reads the three saved tables, draws the figures
(saved in `report/figures/`) and states the insights.

**Run Part B on its own.** Run the two Setup cells at the top, then go on from the setup cell
below. The tables must exist: they do after Part A has run once (and the committed copies are
in the repository).

| Section | Reads |
| :--- | :--- |
| B1 EDA | `pcd_games_clean` |
| B2 SQ1 | `pcd_sq1_groups` |
| B3 SQ2 | `pcd_sq2_groups` |
| B4 SQ3 | `pcd_sq1_groups` and `pcd_sq2_groups`, per genre |

In [63]:
# Part B setup: load the saved tables and prepare the figures folder.
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROCESSED_CSV = ROOT / "data" / "processed" / "csv"
PREFIX = "pcd_"
FIGURES = ROOT / "report" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)


def load(name: str) -> pd.DataFrame:
    """Read one saved output table; stop with a clear message if Part A has not written it."""
    path = PROCESSED_CSV / f"{PREFIX}{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path.relative_to(ROOT)} is missing: run Part A first.")
    return pd.read_csv(path)


games_clean = load("games_clean")
sq1_groups = load("sq1_groups")
sq2_groups = load("sq2_groups")
print(f"games_clean {len(games_clean):,} rows | sq1_groups {len(sq1_groups)} | sq2_groups {len(sq2_groups)}")

games_clean 44,765 rows | sq1_groups 360 | sq2_groups 24


## B1. EDA

## B2. SQ1: how much larger is the reach of games with Simplified Chinese, and in which genres?

## B3. SQ2: does adding Simplified Chinese after launch still increase reach, and in which genres?

## B4. SQ3: for each genre, translate at launch, later, or not a priority?